# NYC Taxi Trip Analytics — Baseline Training

**Objective.** Predict a per-trip target for NYC Yellow Taxi rides from trip distance,
passenger count and cyclically-encoded pickup time.

The target is set by `data.target` in the config and defaults to **`fare_amount`** (USD).
Switch to `trip_duration_minutes` with a single override — the feature set is identical,
so the two runs are directly comparable. Note that `fare_amount` and
`trip_duration_minutes` are **both** excluded from the features: whichever is not the
target is unknown at pickup time, so using it would leak information the model would not
have when serving a live quote.

**Role of this notebook.** Orchestration only. Every piece of logic lives in the
installed `nyc_taxi` package and every hyperparameter lives in `configs/train.yaml`;
this notebook composes them, runs the training, and presents the results.

| Layer | Location |
|---|---|
| Hyperparameters & paths | `configs/train.yaml` |
| ETL (CSV → PostgreSQL) | `src/nyc_taxi/etl.py` |
| Dataset & DataLoaders | `src/nyc_taxi/data.py` |
| Model architecture | `src/nyc_taxi/model.py` |
| Training loop & W&B | `src/nyc_taxi/train.py` |
| SQL schema & analytics | `sql/` |

**Prerequisites**
1. `pip install -e ".[notebooks]"`
2. `POSTGRES_PASSWORD` exported (see `.env.example`).
3. `nyc-taxi-etl` has been run at least once, **or** a parquet cache exists at
   `data/clean_trips.parquet`.

**Execution.** Run all cells top to bottom. There are no out-of-order dependencies.

---
## 1. Environment setup

`autoreload` picks up edits to `src/` without restarting the kernel, so the
notebook and the package never drift apart during development.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from dotenv import load_dotenv

from nyc_taxi.config import load_config, save_config
from nyc_taxi.data import TaxiDataModule
from nyc_taxi.model import build_model
from nyc_taxi.train import Trainer
from nyc_taxi.utils import count_parameters, get_device, seed_everything, setup_logging

warnings.filterwarnings("ignore", category=FutureWarning)
load_dotenv()  # Reads POSTGRES_PASSWORD, WANDB_* etc. from the local .env file.
setup_logging()

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "figure.figsize": (11, 4.5),
        "axes.titleweight": "semibold",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
    }
)
PALETTE = sns.color_palette("crest", n_colors=6)

---
## 2. Configuration

The single source of truth. Nothing below this cell hardcodes a hyperparameter —
to change an experiment, edit `configs/train.yaml` or pass a dotlist override here.

In [ ]:
# NYC_TAXI_OVERRIDES lets CI (and a quick local smoke run) reconfigure this
# notebook without editing it, e.g.
#   NYC_TAXI_OVERRIDES="training.max_epochs=2 data.target=trip_duration_minutes"
OVERRIDES = [token for token in os.environ.get("NYC_TAXI_OVERRIDES", "").split() if token]

cfg = load_config(overrides=OVERRIDES)

seed_everything(cfg.project.seed)
device = get_device(cfg.training.device)

TARGET = cfg.data.target
UNIT = cfg.data.target_unit

print(f"Project        : {cfg.project.name}")
print(f"Seed           : {cfg.project.seed}")
print(f"Device         : {device}")
print(f"Target         : {TARGET} ({UNIT})")
print(f"Overrides      : {OVERRIDES or 'none'}")
print(f"Artifacts      : {cfg.project.artifact_dir}")

In [ ]:
from omegaconf import OmegaConf

print(OmegaConf.to_yaml({k: cfg[k] for k in ("model", "training", "optimizer", "scheduler")}))

---
## 3. Data

`TaxiDataModule.setup()` loads the clean table (parquet cache first, PostgreSQL
otherwise), engineers the cyclical time features, splits train/val/test with the
project seed, and fits the scalers **on the training split only**.

In [ ]:
datamodule = TaxiDataModule(cfg).setup()

print(f"Input features : {datamodule.input_dim}")
print(f"Feature names  : {list(datamodule.spec.names)}")
print(f"Train / Val / Test : "
      f"{len(datamodule.dataset('train')):,} / "
      f"{len(datamodule.dataset('val')):,} / "
      f"{len(datamodule.dataset('test')):,}")

### 3.1 Exploratory view of the target and features

In [ ]:
frame = datamodule.frame

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

sns.histplot(frame[TARGET], bins=80, ax=axes[0], color=PALETTE[2])
axes[0].set(title=f"{TARGET} ({UNIT})", xlabel=UNIT, ylabel="trips")
axes[0].set_xlim(0, float(np.percentile(frame[TARGET], 99)))

sns.histplot(np.log1p(frame[TARGET]), bins=80, ax=axes[1], color=PALETTE[4])
axes[1].set(title="log1p(target) — the space the model learns in",
            xlabel=f"log1p({UNIT})", ylabel="trips")

hourly = frame.groupby("pickup_hour")[TARGET].mean()
axes[2].plot(hourly.index, hourly.to_numpy(), marker="o", color=PALETTE[3])
axes[2].set(title=f"Mean {TARGET} by pickup hour", xlabel="hour of day", ylabel=UNIT)

fig.suptitle("Target distribution and temporal structure", fontsize=13, fontweight="semibold")
fig.tight_layout()
plt.show()

In [ ]:
numeric_cols = [*cfg.data.numerical_features, TARGET]
correlation = frame[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 4.6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="crest", square=True,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Feature correlation")
fig.tight_layout()
plt.show()

---
## 4. Model

`build_model` reads the `model` section of the config and is handed
`datamodule.input_dim`, so the architecture can never silently disagree with the
feature contract.

In [ ]:
model = build_model(cfg, input_dim=datamodule.input_dim)

print(model)
print(f"\nTrainable parameters: {count_parameters(model):,}")

---
## 5. Training

One call. The `Trainer` owns the epoch loop, AMP, gradient clipping, LR
scheduling, early stopping, checkpointing and Weights & Biases logging.

> Set `wandb.enabled=false` (or `WANDB_MODE=offline`) to run without tracking.

In [ ]:
trainer = Trainer(model=model, datamodule=datamodule, cfg=cfg)
history = trainer.fit()

print(f"Completed {len(history)} epoch(s).")

---
## 6. Results

### 6.1 Learning curves

In [ ]:
history_df = pd.DataFrame(history.to_dict()).set_index("epoch")
history_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].plot(history_df.index, history_df["train_loss"], label="train", color=PALETTE[1], marker="o")
axes[0].plot(history_df.index, history_df["val_loss"], label="validation", color=PALETTE[4], marker="o")
axes[0].set(title=f"{cfg.training.criterion.upper()} loss", xlabel="epoch", ylabel="loss")
axes[0].legend(frameon=False)

axes[1].plot(history_df.index, history_df["val_rmse"], color=PALETTE[3], marker="o", label="RMSE")
axes[1].plot(history_df.index, history_df["val_mae"], color=PALETTE[1], marker="s", label="MAE")
best_epoch = int(history_df["val_rmse"].idxmin())
axes[1].axvline(best_epoch, ls="--", lw=1, color="grey")
axes[1].annotate(f"best @ {best_epoch}", (best_epoch, history_df["val_rmse"].min()),
                 textcoords="offset points", xytext=(8, 8), fontsize=9, color="grey")
axes[1].set(title=f"Validation error ({UNIT})", xlabel="epoch", ylabel=UNIT)
axes[1].legend(frameon=False)

axes[2].plot(history_df.index, history_df["learning_rate"], color=PALETTE[5], marker="o")
axes[2].set(title=f"Learning rate ({cfg.scheduler.name})", xlabel="epoch", ylabel="lr")
axes[2].set_yscale("log")

fig.suptitle("Training dynamics", fontsize=13, fontweight="semibold")
fig.tight_layout()
plt.show()

### 6.2 Held-out test evaluation

`evaluate` restores the best checkpoint first, so these numbers describe the
selected model rather than the final epoch.

In [ ]:
test_metrics = trainer.evaluate(split="test", restore_best=True)

pd.DataFrame(
    [
        {"metric": f"RMSE ({UNIT})", "value": test_metrics["rmse"]},
        {"metric": f"MAE ({UNIT})", "value": test_metrics["mae"]},
        {"metric": "R²", "value": test_metrics["r2"]},
        {"metric": f"{cfg.training.criterion.upper()} loss", "value": test_metrics["loss"]},
    ]
).style.format({"value": "{:.4f}"}).hide(axis="index").set_caption("Test-set performance")

### 6.3 Prediction quality

In [ ]:
y_pred, y_true = trainer.predict(datamodule.test_dataloader())
residuals = y_pred - y_true

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

sample = np.random.default_rng(cfg.project.seed).choice(len(y_true), size=min(20_000, len(y_true)), replace=False)
axes[0].scatter(y_true[sample], y_pred[sample], s=3, alpha=0.15, color=PALETTE[3], edgecolors="none")
limit = float(np.percentile(y_true, 99.5))
axes[0].plot([0, limit], [0, limit], ls="--", lw=1.2, color="black", label="perfect")
axes[0].set(title="Predicted vs. actual", xlabel=f"actual ({UNIT})",
            ylabel=f"predicted ({UNIT})", xlim=(0, limit), ylim=(0, limit))
axes[0].legend(frameon=False)

sns.histplot(residuals, bins=100, ax=axes[1], color=PALETTE[2])
axes[1].axvline(0, ls="--", lw=1.2, color="black")
axes[1].set(title=f"Residuals (μ={residuals.mean():.2f}, σ={residuals.std():.2f})",
            xlabel=f"predicted − actual ({UNIT})", ylabel="trips")
axes[1].set_xlim(np.percentile(residuals, 0.5), np.percentile(residuals, 99.5))

bins = np.percentile(y_true, np.linspace(0, 100, 11))
bin_index = np.clip(np.digitize(y_true, bins[1:-1]), 0, 9)
bin_mae = [np.abs(residuals[bin_index == i]).mean() for i in range(10)]
axes[2].bar(range(1, 11), bin_mae, color=PALETTE[4])
axes[2].set(title=f"MAE by actual-{TARGET} decile", xlabel=f"decile of true {TARGET}",
            ylabel=f"MAE ({UNIT})")
axes[2].set_xticks(range(1, 11))

fig.suptitle("Test-set error analysis", fontsize=13, fontweight="semibold")
fig.tight_layout()
plt.show()

### 6.4 Comparison against the SQL heuristic baseline

`sql/02_analytics.sql` (query 6) predicts each trip's duration as the mean of its
`(dayofweek, hour)` bucket. The learned model must beat that to justify itself.

In [ ]:
RMSE_COL, MAE_COL = f"RMSE ({UNIT})", f"MAE ({UNIT})"

bucket_means = frame.groupby(["pickup_dayofweek", "pickup_hour"])[TARGET].transform("mean")
baseline_rmse = float(np.sqrt(np.mean((frame[TARGET] - bucket_means) ** 2)))
baseline_mae = float(np.mean(np.abs(frame[TARGET] - bucket_means)))

comparison = pd.DataFrame(
    {
        "model": ["(dayofweek, hour) mean", cfg.model.name],
        RMSE_COL: [baseline_rmse, test_metrics["rmse"]],
        MAE_COL: [baseline_mae, test_metrics["mae"]],
    }
)
comparison["RMSE improvement"] = 1 - comparison[RMSE_COL] / baseline_rmse

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(comparison["model"], comparison[RMSE_COL], color=[PALETTE[1], PALETTE[4]])
for y, value in enumerate(comparison[RMSE_COL]):
    ax.text(value, y, f"  {value:.2f}", va="center", fontsize=10)
ax.set(title=f"RMSE ({UNIT}) — lower is better", xlabel=UNIT)
ax.invert_yaxis()
fig.tight_layout()
plt.show()

comparison.style.format({RMSE_COL: "{:.3f}", MAE_COL: "{:.3f}",
                         "RMSE improvement": "{:+.1%}"}).hide(axis="index")

---
## 7. Artifacts

The resolved config is written next to the checkpoint so this exact run can be
reproduced from disk alone.

In [ ]:
checkpoint_path = trainer.checkpoint_path
config_path = save_config(cfg, Path(cfg.training.checkpoint_dir) / "resolved_config.yaml")

print(f"Best checkpoint : {checkpoint_path}  ({checkpoint_path.stat().st_size / 1e6:.2f} MB)")
print(f"Resolved config : {config_path}")

---
## 8. Summary & next steps

**Outcome.** The residual MLP is compared against the `(dayofweek, hour)` mean
heuristic on a held-out test split; the tables and figures above report RMSE, MAE
and R² in the target's own unit, all restored from the best checkpoint rather than
the last epoch.

Run `python scripts/compare_targets.py` to benchmark both candidate targets
side by side from the same feature set.

**Reproduce headlessly**

```bash
nyc-taxi-etl                                  # CSV → PostgreSQL
nyc-taxi-train training.max_epochs=30         # this notebook, without the plots
```

**Next experiments**
- Add pickup/dropoff geography (`PULocationID`/`DOLocationID`) — currently dropped by the ETL projection.
- Sweep `model.hidden_dims`, `model.dropout` and `optimizer.lr` with a W&B sweep.
- Benchmark against gradient boosting to confirm the deep model earns its complexity.